In [ ]:
# CoinMarketCap Scraping Notebook
# FULL TORD UNIVERSE • 24 MONTHS DAILY • ONE MERGED OUTPUT
# SAFE AGAINST OHLCV 10,000-DATAPOINT LIMIT

import os
import time
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import requests
from dotenv import load_dotenv

# ============================================================
# 0) Setup
# ============================================================
load_dotenv()

API_KEY = os.getenv("CMC_API_KEY")
if not API_KEY:
    raise RuntimeError("Set CMC_API_KEY in your environment (.env is fine).")

BASE_URL = "https://pro-api.coinmarketcap.com"
HEADERS = {
    "Accept": "application/json",
    "X-CMC_PRO_API_KEY": API_KEY,
}

TORD_CLEAN = Path("../processed/tord_clean_min.csv")
OUTDIR     = Path("../raw")

# Startup-plan-safe settings
CHUNK_SIZE = 40
SLEEP_BETWEEN_CALLS = 0.35

# Full ~24 months daily history
END_DATE = datetime.utcnow().date()
START_DATE = END_DATE - timedelta(days=730)

# IMPORTANT:
# CMC OHLCV historical has a max datapoint cap per request.
# So we split the 24 months into smaller windows.
OHLCV_WINDOW_DAYS = 180   # safe: 40 ids * 180 days = 7200 datapoints

print("Date window:", START_DATE.isoformat(), "to", END_DATE.isoformat())
print("Output dir:", OUTDIR.resolve())

# ============================================================
# 1) Load TORD and build the token universe
# ============================================================
tord = pd.read_csv(TORD_CLEAN)

if "Coinmarketcap_identifier" not in tord.columns:
    raise KeyError("TORD is missing Coinmarketcap_identifier")

tord_ids = tord[tord["Coinmarketcap_identifier"].notna()].copy()
tord_ids["cmc_id"] = pd.to_numeric(tord_ids["Coinmarketcap_identifier"], errors="coerce")
tord_ids = tord_ids[tord_ids["cmc_id"].notna()].copy()
tord_ids["cmc_id"] = tord_ids["cmc_id"].astype(int)

unique_ids = sorted(tord_ids["cmc_id"].unique().tolist())

print("TORD rows with CMC id:", len(tord_ids))
print("Unique CMC ids:", len(unique_ids))

# ============================================================
# 2) Helpers
# ============================================================
def chunks(lst, n=40):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def make_date_windows(start_date, end_date, window_days=180):
    """
    Inclusive windows from start_date to end_date.
    """
    windows = []
    cur_start = start_date
    while cur_start <= end_date:
        cur_end = min(cur_start + timedelta(days=window_days - 1), end_date)
        windows.append((cur_start, cur_end))
        cur_start = cur_end + timedelta(days=1)
    return windows

def cmc_get(url, params=None, retries=7, base_sleep=1.0):
    """
    Defensive GET with exponential backoff for 429/5xx.
    """
    params = params or {}
    for k in range(retries):
        r = requests.get(url, headers=HEADERS, params=params, timeout=90)

        if r.status_code == 200:
            return r.json()

        if r.status_code in (429, 500, 502, 503, 504):
            sleep_s = base_sleep * (2 ** k)
            print(f"Retryable error {r.status_code}; sleeping {sleep_s:.1f}s")
            time.sleep(sleep_s)
            continue

        raise RuntimeError(f"CMC {r.status_code}: {r.text}")

    raise RuntimeError(f"CMC failed after {retries} retries. Params={params}")

def safe_to_numeric(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ============================================================
# 3) Pull DAILY OHLCV history
# ============================================================
OHLCV_HIST = f"{BASE_URL}/v1/cryptocurrency/ohlcv/historical"

def parse_ohlcv(payload):
    data = payload.get("data", {})

    # Single-object response shape
    if isinstance(data, dict) and "quotes" in data:
        data = {str(data.get("id")): data}

    rows = []
    for _, obj in (data or {}).items():
        cid = obj.get("id")
        if cid is None:
            continue
        cid = int(cid)

        for q in obj.get("quotes") or []:
            usd = ((q.get("quote") or {}).get("USD") or {})
            ts = usd.get("timestamp") or q.get("time_close") or q.get("time_open")
            if not ts:
                continue

            d = pd.to_datetime(ts, utc=True, errors="coerce")
            if pd.isna(d):
                continue

            rows.append({
                "cmc_id": cid,
                "date": d.date(),
                "open": usd.get("open"),
                "high": usd.get("high"),
                "low": usd.get("low"),
                "close": usd.get("close"),
                "volume": usd.get("volume"),
                "market_cap": usd.get("market_cap"),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out = safe_to_numeric(out, ["open", "high", "low", "close", "volume", "market_cap"])
    return out

date_windows = make_date_windows(START_DATE, END_DATE, OHLCV_WINDOW_DAYS)
print("OHLCV date windows:", len(date_windows))
for w in date_windows:
    print(" -", w[0].isoformat(), "to", w[1].isoformat())

ohlcv_parts = []
request_counter = 0

for chunk_i, ids_chunk in enumerate(chunks(unique_ids, CHUNK_SIZE), start=1):
    id_string = ",".join(map(str, ids_chunk))

    for window_i, (win_start, win_end) in enumerate(date_windows, start=1):
        params = {
            "id": id_string,
            "time_period": "daily",
            "time_start": win_start.isoformat(),
            "time_end": win_end.isoformat(),
            "convert": "USD",
            "skip_invalid": "true",
        }

        payload = cmc_get(OHLCV_HIST, params)
        dfp = parse_ohlcv(payload)

        if not dfp.empty:
            ohlcv_parts.append(dfp)

        request_counter += 1
        print(
            f"Fetched OHLCV request {request_counter} | "
            f"id chunk {chunk_i} | window {window_i}/{len(date_windows)} | "
            f"{win_start.isoformat()} to {win_end.isoformat()}"
        )
        time.sleep(SLEEP_BETWEEN_CALLS)

ohlcv = pd.concat(ohlcv_parts, ignore_index=True) if ohlcv_parts else pd.DataFrame()

if not ohlcv.empty:
    ohlcv = (
        ohlcv
        .drop_duplicates(["cmc_id", "date"])
        .sort_values(["cmc_id", "date"])
        .reset_index(drop=True)
    )

print("OHLCV shape:", ohlcv.shape)
print("OHLCV unique tokens:", ohlcv["cmc_id"].nunique() if not ohlcv.empty else 0)

# ============================================================
# 4) Pull token metadata
# ============================================================
INFO_URL = f"{BASE_URL}/v1/cryptocurrency/info"

def parse_info(payload):
    data = payload.get("data", {}) or {}
    rows = []

    for k, obj in data.items():
        cid = obj.get("id") or k
        cid = int(cid)

        platform = obj.get("platform") or {}
        tags = obj.get("tags") or []
        description = obj.get("description") or ""

        rows.append({
            "cmc_id": cid,
            "name": obj.get("name"),
            "symbol": obj.get("symbol"),
            "slug": obj.get("slug"),
            "date_added": obj.get("date_added"),
            "category": obj.get("category"),
            "tags": "|".join(tags) if isinstance(tags, list) else None,
            "platform_id": platform.get("id"),
            "platform_name": platform.get("name"),
            "platform_symbol": platform.get("symbol"),
            "platform_slug": platform.get("slug"),
            "platform_token_address": platform.get("token_address"),
            "is_hidden": obj.get("is_hidden"),
            "description_len": len(description),
        })

    return pd.DataFrame(rows)

meta_parts = []
for i, ids_chunk in enumerate(chunks(unique_ids, CHUNK_SIZE), start=1):
    params = {
        "id": ",".join(map(str, ids_chunk)),
        "skip_invalid": "true",
        "aux": "urls,logo,description,tags,platform,date_added,notice,status",
    }

    payload = cmc_get(INFO_URL, params)
    dfp = parse_info(payload)

    if not dfp.empty:
        meta_parts.append(dfp)

    print(f"Fetched INFO chunk {i}")
    time.sleep(SLEEP_BETWEEN_CALLS)

meta = pd.concat(meta_parts, ignore_index=True) if meta_parts else pd.DataFrame()

if not meta.empty:
    meta = meta.drop_duplicates(["cmc_id"]).reset_index(drop=True)

print("Metadata shape:", meta.shape)

# ============================================================
# 5) Pull latest quote snapshot
# ============================================================
QUOTES_URL = f"{BASE_URL}/v2/cryptocurrency/quotes/latest"

def parse_quotes_latest(payload):
    data = payload.get("data", {}) or {}
    rows = []

    for _, obj in data.items():
        items = obj if isinstance(obj, list) else [obj]

        for x in items:
            cid = x.get("id")
            if cid is None:
                continue
            cid = int(cid)

            usd = ((x.get("quote") or {}).get("USD") or {})

            rows.append({
                "cmc_id": cid,
                "price_latest": usd.get("price"),
                "volume_24h_latest": usd.get("volume_24h"),
                "market_cap_latest": usd.get("market_cap"),
                "circulating_supply": x.get("circulating_supply"),
                "total_supply": x.get("total_supply"),
                "max_supply": x.get("max_supply"),
                "cmc_rank": x.get("cmc_rank"),
                "num_market_pairs": x.get("num_market_pairs"),
                "last_updated_latest": usd.get("last_updated"),
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out = safe_to_numeric(out, [
        "price_latest",
        "volume_24h_latest",
        "market_cap_latest",
        "circulating_supply",
        "total_supply",
        "max_supply",
        "cmc_rank",
        "num_market_pairs",
    ])
    return out

quote_parts = []
for i, ids_chunk in enumerate(chunks(unique_ids, CHUNK_SIZE), start=1):
    params = {
        "id": ",".join(map(str, ids_chunk)),
        "convert": "USD",
        "skip_invalid": "true",
    }

    payload = cmc_get(QUOTES_URL, params)
    dfp = parse_quotes_latest(payload)

    if not dfp.empty:
        quote_parts.append(dfp)

    print(f"Fetched QUOTES chunk {i}")
    time.sleep(SLEEP_BETWEEN_CALLS)

quotes_latest = pd.concat(quote_parts, ignore_index=True) if quote_parts else pd.DataFrame()

if not quotes_latest.empty:
    quotes_latest = quotes_latest.drop_duplicates(["cmc_id"]).reset_index(drop=True)

print("Latest quotes shape:", quotes_latest.shape)

# ============================================================
# 6) Merge into ONE flat daily file
# ============================================================
if ohlcv.empty:
    raise RuntimeError("No OHLCV data was returned. Check your API key, rate limits, or TORD cmc_id universe.")

merged = ohlcv.copy()

if not meta.empty:
    merged = merged.merge(meta, on="cmc_id", how="left")

if not quotes_latest.empty:
    merged = merged.merge(quotes_latest, on="cmc_id", how="left")

# Optional: merge one TORD row per cmc_id
# Uncomment only if you want TORD columns repeated on each daily row
#
# tord_one_per_cmc = (
#     tord_ids.sort_values("cmc_id")
#            .drop_duplicates("cmc_id")
#            .drop(columns=["Coinmarketcap_identifier"])
# )
# merged = merged.merge(tord_one_per_cmc, on="cmc_id", how="left")

merged = merged.sort_values(["cmc_id", "date"]).reset_index(drop=True)

# ============================================================
# 7) Save outputs
# ============================================================
out_csv = OUTDIR / "cmc_tord_daily_merged.csv"
merged.to_csv(out_csv, index=False)

print("Saved CSV:", out_csv)
print("Merged shape:", merged.shape)
print("Merged unique tokens:", merged["cmc_id"].nunique())
print("Merged date range:", merged["date"].min().date(), "to", merged["date"].max().date())

# Optional helper files
ohlcv.to_csv(OUTDIR / "cmc_ohlcv_daily_raw.csv", index=False)
if not meta.empty:
    meta.to_csv(OUTDIR / "cmc_token_metadata_raw.csv", index=False)
if not quotes_latest.empty:
    quotes_latest.to_csv(OUTDIR / "cmc_token_quotes_latest_raw.csv", index=False)

print("Done.")

C:\Users\paolina.gocheva\AppData\Local\Temp\ipykernel_34412\1808075451.py:38: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  END_DATE = datetime.utcnow().date()


Date window: 2024-03-31 to 2026-03-31
Output dir: C:\Users\paolina.gocheva\OneDrive - Solactive AG\Desktop\me\datasets\outputs_cmc
TORD rows with CMC id: 3469
Unique CMC ids: 3428
OHLCV date windows: 5
 - 2024-03-31 to 2024-09-26
 - 2024-09-27 to 2025-03-25
 - 2025-03-26 to 2025-09-21
 - 2025-09-22 to 2026-03-20
 - 2026-03-21 to 2026-03-31
Fetched OHLCV request 1 | id chunk 1 | window 1/5 | 2024-03-31 to 2024-09-26
Fetched OHLCV request 2 | id chunk 1 | window 2/5 | 2024-09-27 to 2025-03-25
Fetched OHLCV request 3 | id chunk 1 | window 3/5 | 2025-03-26 to 2025-09-21
Fetched OHLCV request 4 | id chunk 1 | window 4/5 | 2025-09-22 to 2026-03-20
Fetched OHLCV request 5 | id chunk 1 | window 5/5 | 2026-03-21 to 2026-03-31
Fetched OHLCV request 6 | id chunk 2 | window 1/5 | 2024-03-31 to 2024-09-26
Fetched OHLCV request 7 | id chunk 2 | window 2/5 | 2024-09-27 to 2025-03-25
Fetched OHLCV request 8 | id chunk 2 | window 3/5 | 2025-03-26 to 2025-09-21
Fetched OHLCV request 9 | id chunk 2 | win

In [3]:
import pandas as pd

df = pd.read_csv("outputs_cmc/cmc_tord_daily_merged.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Unique tokens/coins:", df["cmc_id"].nunique())
print("Time period:", df["date"].min().date(), "to", df["date"].max().date())
print("Total datapoints:", len(df))
print("Average observations per token:", round(df.groupby("cmc_id")["date"].nunique().mean(), 2))

Unique tokens/coins: 1853
Time period: 2024-04-01 to 2026-03-30
Total datapoints: 1026525
Average observations per token: 553.98


In [4]:
import pandas as pd

tord = pd.read_csv("tord_clean_min.csv")

# Basic structure
print("Shape:", tord.shape)
print("\nColumns:")
print(tord.columns.tolist())

print("\nDtypes:")
print(tord.dtypes)

print("\nFirst 5 rows:")
display(tord.head())

print("\nMissing values per column:")
display(tord.isna().sum().sort_values(ascending=False))

print("\nSummary stats:")
display(tord.describe(include="all").T)

Shape: (10807, 56)

Columns:
['id', 'name', 'token', 'country', 'is_ico', 'is_ieo', 'is_sto', 'ico_start', 'ico_end', 'price_usd', 'raised_usd', 'distributed_in_ico', 'sold_tokens', 'token_for_sale', 'whitelist', 'kyc', 'bonus', 'restricted_areas', 'min_investment', 'bounty', 'mvp', 'pre_ico_start', 'pre_ico_end', 'pre_ico_price_usd', 'platform', 'accepting', 'link_white_paper', 'linkedin_link', 'github_link', 'website', 'rating', 'teamsize', 'Coinmarketcap_identifier', 'ERC20', 'contract_address', 'Source', 'distributed_in_ico_frac', 'min_investment_raw', 'min_inv_clean', 'min_inv_pairs', 'min_inv_min_value', 'min_inv_min_unit', 'min_inv_usd_value', 'flag_min_inv_multi', 'flag_min_inv_ratio', 'flag_min_inv_equals', 'flag_min_inv_missinglike', 'ico_duration_days', 'has_presale', 'has_whitepaper', 'has_github', 'has_linkedin', 'has_website', 'flag_suspicious_ico_start', 'flag_negative_price', 'flag_negative_raised']

Dtypes:
id                             int64
name                     

,id,name,token,country,is_ico,is_ieo,is_sto,ico_start,ico_end,price_usd,...,flag_min_inv_missinglike,ico_duration_days,has_presale,has_whitepaper,has_github,has_linkedin,has_website,flag_suspicious_ico_start,flag_negative_price,flag_negative_raised
0,1,WINSSHI,WNS,India,1.0,0.0,0.0,2020-08-10,2020-12-31,0.01,...,0,143.0,0,0,0,1,0,0,0,0
1,2,Tycoon,TYCOONTOKEN/TYC,Cyprus,1.0,0.0,0.0,2020-08-01,2020-12-31,0.10,...,1,152.0,1,1,1,1,1,0,0,0
2,3,Mindsync,MAI,UK,1.0,0.0,0.0,2019-03-01,2020-12-31,0.14,...,0,671.0,1,1,0,1,0,0,0,0
3,4,PointPay,PXP,UK,1.0,0.0,0.0,2020-06-25,2021-01-31,0.10,...,0,220.0,0,1,0,1,0,0,0,0
4,5,LOHN,LOHN,Seychelles,1.0,0.0,0.0,NaN,NaN,0.06,...,0,NaN,0,1,1,1,1,0,0,0



Missing values per column:


sold_tokens                  10615
min_inv_usd_value            10195
restricted_areas              8801
min_inv_pairs                 8758
min_inv_min_unit              8758
min_inv_min_value             8758
min_investment                8732
min_inv_clean                 8732
min_investment_raw            8732
pre_ico_price_usd             8702
raised_usd                    8503
contract_address              8238
pre_ico_end                   8102
pre_ico_start                 8090
github_link                   7430
Coinmarketcap_identifier      7338
mvp                           7224
whitelist                     6844
linkedin_link                 6452
ico_duration_days             5329
ico_end                       5310
distributed_in_ico_frac       5178
distributed_in_ico            5178
ico_start                     5171
teamsize                      5049
token_for_sale                4926
is_ieo                        4392
is_sto                        4392
is_ico              


Summary stats:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,10807.0,NaN,NaN,NaN,5412.831313,3123.359062,1.0,2709.5,5415.0,8117.5,10819.0
name,10807,10792,Hero,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
token,10668,8658,SMT,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,7148,213,USA,889,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_ico,6415.0,NaN,NaN,NaN,0.931878,0.251974,0.0,1.0,1.0,1.0,1.0
is_ieo,6415.0,NaN,NaN,NaN,0.068122,0.251974,0.0,0.0,0.0,0.0,1.0
is_sto,6415.0,NaN,NaN,NaN,0.013874,0.116976,0.0,0.0,0.0,0.0,1.0
ico_start,5636,1096,2018-03-01,88,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ico_end,5497,1127,2018-12-31,113,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_usd,7183.0,NaN,NaN,NaN,115706.190918,9775355.40435,0.0,0.18015,1.0,2.0,828486022.0
